# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [2]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:item in an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [10]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'])
orders_df = lines
print(lines)
assert len(lines) == 4
assert lines['qty'].sum() == 7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [11]:
# TODO
vendors_df = pd.json_normalize(vendors_json)
joined = lines.merge(
    vendors_df,
    on='vendor_id',
    how='left',
    indicator=True)
joined_df = joined
print(joined)
unmatched = joined[joined['_merge'] == 'left_only']
print(unmatched['vendor_id'].unique())
print(unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
['V-07']
3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [12]:
# TODO
joined_df['name'] = joined_df['name'].fillna('Unknown Vendor')
result = joined_df.groupby('name')['qty'].sum().reset_index()
print(result)

             name  qty
0       Cav Merch    1
1    Hoos Burgers    3
2  Unknown Vendor    3


I keep the unmatched vendor as "Unknown Vendor" because its quantity still need to be included in the total.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [13]:
# TODO
zone_totals = joined_df.groupby('zone')['qty'].sum().reset_index()
print(zone_totals)
print(zone_totals['qty'].sum())
print(joined_df['qty'].sum() - zone_totals['qty'].sum())

  zone  qty
0    A    4
4
3


Zone A sold the most items with 4 units. The total is 4 because vendor V-07 has no zone information.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [14]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_fixed = []
for order in ragged_json:
    if 'lines' not in order:
        order['lines'] = []
    ragged_fixed.append(order)
ragged_df = pd.json_normalize(
    ragged_fixed,
    record_path='lines',
    meta=['order', 'vendor_id'])
print(ragged_df)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [15]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item makes people easier to count and compare items. It will be easy to find the total quantity sold by each vendor. However, with one row per order, people will have to separate the items first.

b)A missing quantity means people don't know how many items were sold, while zero means no items were sold. If people treat missing values as zero, the totals and analysis will be inaccurate because unknown values are regarded as actual zero sales.